In [2]:
import numpy as np
import pandas as pd



In [3]:
df = pd.read_csv('../data/raw/financial_fraud_detection_dataset.csv')



In [19]:
df.shape

(5000000, 19)

The dataset contains **5 million financial transactions**.  

If there are just a few rows with missing values, they can be safely dropped without significantly affecting the overall data distribution.

In [4]:
df.head()

,transaction_id,timestamp,sender_account,receiver_account,amount,transaction_type,merchant_category,location,device_used,is_fraud,fraud_type,time_since_last_transaction,spending_deviation_score,velocity_score,geo_anomaly_score,payment_channel,ip_address,device_hash
0,T100000,2023-08-22T09:22:43.516168,ACC877572,ACC388389,343.78,withdrawal,utilities,Tokyo,mobile,False,NaN,NaN,-0.21,3,0.22,card,13.101.214.112,D8536477
1,T100001,2023-08-04T01:58:02.606711,ACC895667,ACC944962,419.65,withdrawal,online,Toronto,atm,False,NaN,NaN,-0.14,7,0.96,ACH,172.52.47.194,D2622631
2,T100002,2023-05-12T11:39:33.742963,ACC733052,ACC377370,2773.86,deposit,other,London,pos,False,NaN,NaN,-1.78,20,0.89,card,185.98.35.23,D4823498
3,T100003,2023-10-10T06:04:43.195112,ACC996865,ACC344098,1666.22,deposit,online,Sydney,pos,False,NaN,NaN,-0.60,6,0.37,wire_transfer,107.136.36.87,D9961380
4,T100004,2023-09-24T08:09:02.700162,ACC584714,ACC497887,24.43,transfer,utilities,Toronto,mobile,False,NaN,NaN,0.79,13,0.27,ACH,108.161.108.255,D7637601


In [5]:
df.tail()

,transaction_id,timestamp,sender_account,receiver_account,amount,transaction_type,merchant_category,location,device_used,is_fraud,fraud_type,time_since_last_transaction,spending_deviation_score,velocity_score,geo_anomaly_score,payment_channel,ip_address,device_hash
4999995,T5099995,2023-11-17T23:20:29.746144,ACC597319,ACC749300,10.87,withdrawal,retail,Toronto,atm,False,NaN,1416.524233,-0.14,17,0.18,UPI,243.92.38.163,D4439579
4999996,T5099996,2023-09-23T11:23:20.659686,ACC749625,ACC709783,181.40,payment,grocery,Sydney,atm,False,NaN,999.089702,-1.79,4,0.58,wire_transfer,28.252.18.249,D5029311
4999997,T5099997,2023-11-18T00:52:34.527092,ACC629492,ACC680736,12.54,payment,utilities,New York,mobile,False,NaN,3871.584025,-0.30,6,0.99,card,111.199.174.121,D6333607
4999998,T5099998,2023-03-25T04:32:13.609837,ACC984720,ACC296935,376.29,deposit,restaurant,Dubai,pos,False,NaN,-4096.765453,-1.43,5,0.32,wire_transfer,221.110.215.14,D1551203
4999999,T5099999,2023-09-02T04:34:34.583803,ACC120255,ACC440137,7.27,transfer,grocery,Sydney,atm,False,NaN,5257.349021,0.08,14,0.40,wire_transfer,246.68.126.184,D1505627


The dataset is NOT sorted by the `timestamp` column. 
Evidence: In df.head(), the second row (2023-08-04) is earlier than the first row (2023-08-22). 
In df.tail(), timestamps jump around (2023-11-17, 2023-09-23, 2023-11-18, 2023-03-25, 2023-09-02). 
It appears the data is sorted only by `transaction_id`.

In [6]:
df.sample(5)

,transaction_id,timestamp,sender_account,receiver_account,amount,transaction_type,merchant_category,location,device_used,is_fraud,fraud_type,time_since_last_transaction,spending_deviation_score,velocity_score,geo_anomaly_score,payment_channel,ip_address,device_hash
935542,T1035542,2023-06-01T20:54:58.844394,ACC739135,ACC218644,59.46,payment,travel,Dubai,web,False,NaN,-1094.626612,-0.01,13,0.09,wire_transfer,29.142.181.177,D5392993
3236054,T3336054,2023-10-25T14:50:32.686059,ACC812525,ACC513174,169.94,withdrawal,other,New York,pos,False,NaN,-1084.350728,-1.29,16,0.60,wire_transfer,145.134.216.116,D6353713
1521989,T1621989,2023-08-20T02:37:42.843161,ACC814748,ACC871258,96.83,payment,restaurant,New York,pos,False,NaN,858.831043,-0.08,2,0.87,UPI,152.43.131.64,D7966134
2694917,T2794917,2023-04-07T23:55:08.154974,ACC981713,ACC395727,525.10,withdrawal,online,Singapore,pos,False,NaN,-881.787099,0.69,15,0.47,card,207.131.241.11,D5132534
740902,T840902,2023-04-06T14:16:35.077300,ACC511860,ACC139467,32.51,transfer,restaurant,Dubai,atm,False,NaN,NaN,0.72,14,0.94,card,34.38.160.200,D6057101


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000000 entries, 0 to 4999999
Data columns (total 18 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   transaction_id               str    
 1   timestamp                    str    
 2   sender_account               str    
 3   receiver_account             str    
 4   amount                       float64
 5   transaction_type             str    
 6   merchant_category            str    
 7   location                     str    
 8   device_used                  str    
 9   is_fraud                     bool   
 10  fraud_type                   str    
 11  time_since_last_transaction  float64
 12  spending_deviation_score     float64
 13  velocity_score               int64  
 14  geo_anomaly_score            float64
 15  payment_channel              str    
 16  ip_address                   str    
 17  device_hash                  str    
dtypes: bool(1), float64(4), int64(1), str(12)
memory usag

## Feature Selection and Target Variable

We use **`is_fraud`** as the target variable, as our goal is to predict whether a transaction is fraudulent.  

We remove the columns **`transaction_id`** and **`fraud_type`** from the model: the former serves only as an identifier, and including the latter would introduce data leakage.  

The behavioral and statistical features — **`amount`**, **`time_since_last_transaction`**, **`spending_deviation_score`**, **`velocity_score`**, and **`geo_anomaly_score`** — are likely highly informative for detecting fraudulent transactions.  

Contextual transaction features — **`transaction_type`**, **`merchant_category`**, **`payment_channel`**, and **`device_used`** — are also predictive, but must be encoded as categorical variables (e.g., using one-hot encoding or ordinal encoding) before being used by the model.  

Other columns — **`timestamp`**, **`location`**, **`sender_account`**, **`receiver_account`**, **`ip_address`**, and **`device_hash`** — are not useful in their raw string form, but can be transformed into highly predictive features through feature engineering.  

In general, several columns that appear uninformative as raw strings can yield significant value when used to derive engineered features.

In [8]:
df.isna().sum()

transaction_id                       0
timestamp                            0
sender_account                       0
receiver_account                     0
amount                               0
transaction_type                     0
merchant_category                    0
location                             0
device_used                          0
is_fraud                             0
fraud_type                     4820447
time_since_last_transaction     896513
spending_deviation_score             0
velocity_score                       0
geo_anomaly_score                    0
payment_channel                      0
ip_address                           0
device_hash                          0
dtype: int64

In [9]:
percentage_of_missing_values_in_time_since_last_transaction = df['time_since_last_transaction'].isnull().mean() * 100
print(f"Percentage of missing values in 'time_since_last_transaction': {percentage_of_missing_values_in_time_since_last_transaction:.2f}%")

Percentage of missing values in 'time_since_last_transaction': 17.93%


# Handling Missing Values

Most columns do not contain missing values.  

However, the column **`time_since_last_transaction`** has approximately **18% missing values**, which is understandable since not every transaction has a previous transaction for the corresponding user.  

We will impute these missing values **only after splitting the data into training and test sets** to avoid introducing data leakage.  

In the meantime, we can create a **missing flag** — a new column indicating whether a value is missing — to preserve this information for the model.

In [10]:
# Creating missing flags for 'time_since_last_transaction'
df['time_since_last_transaction_missing'] = df['time_since_last_transaction'].isnull().astype(int)

In [11]:
dup_counts = df.apply(lambda col: col.duplicated(keep=False).sum()) # keep=False → marks all duplicates (including the first occurrence) as True.
print(dup_counts)

transaction_id                               0
timestamp                                    4
sender_account                         4980790
receiver_account                       4980744
amount                                 4977640
transaction_type                       5000000
merchant_category                      5000000
location                               5000000
device_used                            5000000
is_fraud                               5000000
fraud_type                             5000000
time_since_last_transaction             896513
spending_deviation_score               4999956
velocity_score                         5000000
geo_anomaly_score                      5000000
payment_channel                        5000000
ip_address                                5864
device_hash                            2132053
time_since_last_transaction_missing    5000000
dtype: int64


# Duplicate Checking – Interpretation

- **transaction_id:** 0 duplicates  
  - ✅ Unique identifier, perfect.

- **timestamp:** 4 duplicates  
  - Minor duplication, likely simultaneous transactions. Worth a brief check.

- **sender_account:** 4,980,790 duplicates  
- **receiver_account:** 4,980,744 duplicates  
  - Expected; accounts appear in multiple transactions. Useful for aggregated features (e.g., transaction counts, average amount).

- **amount:** 4,977,640 duplicates  
  - Many transactions share the same amount; normal for financial data.

- **transaction_type, merchant_category, location, device_used, is_fraud, fraud_type, velocity_score, payment_channel, geo_anomaly_score:** ~5,000,000 duplicates  
  - High duplication expected for categorical/constant columns; suitable for modeling and feature engineering.

- **time_since_last_transaction:** 896,513 duplicates  
  - Repeated time gaps are reasonable; can be used numerically after imputing missing values.

- **spending_deviation_score:** 4,999,956 duplicates  
  - Mostly repeated values; may require scaling/normalization.

- **ip_address:** 5,864 duplicates  
  - Some IPs used multiple times; can help detect shared or suspicious IPs.

- **device_hash:** 2,132,053 duplicates  
  - Shared across transactions; useful for identifying reused devices or fraud clusters.

**Summary:**  
- Only `transaction_id` is strictly unique; all others have expected duplicates.  
- Duplication in account, device, and categorical columns is normal and **key for feature engineering**.  
- No major data quality issues

In [12]:
fraud_counts = df['is_fraud'].value_counts()
fraud_counts 

is_fraud
False    4820447
True      179553
Name: count, dtype: int64

In [13]:
fraud_proportions = df['is_fraud'].value_counts(normalize=True)
fraud_proportions

is_fraud
False    0.964089
True     0.035911
Name: proportion, dtype: float64

In the following cell, we convert the **`timestamp`** column from a string representation to a datetime format.

In [14]:
df.head()["timestamp"]

0    2023-08-22T09:22:43.516168
1    2023-08-04T01:58:02.606711
2    2023-05-12T11:39:33.742963
3    2023-10-10T06:04:43.195112
4    2023-09-24T08:09:02.700162
Name: timestamp, dtype: str

In [16]:
df["timestamp"] = pd.to_datetime(df["timestamp"], errors='coerce')

In [17]:
bad_rows = df[df["timestamp"].isna()]
print(bad_rows)

        transaction_id timestamp sender_account receiver_account  amount  \
1891874       T1991874       NaT      ACC375307        ACC630432  195.84   
2833396       T2933396       NaT      ACC969484        ACC369350    4.55   
3214871       T3314871       NaT      ACC444490        ACC483161   28.20   

        transaction_type merchant_category   location device_used  is_fraud  \
1891874          deposit           grocery      Dubai         web     False   
2833396          payment            travel      Dubai         web     False   
3214871         transfer            online  Singapore         atm     False   

        fraud_type  time_since_last_transaction  spending_deviation_score  \
1891874        NaN                  4429.861012                      0.87   
2833396        NaN                  -352.835279                      1.20   
3214871        NaN                  6481.381027                     -1.32   

         velocity_score  geo_anomaly_score payment_channel      ip_ad

Since this transformation introduced **3 new NaN rows**, and as previously noted, such a small number has minimal impact, we will simply drop them.

In [20]:
df = df.dropna(subset=["timestamp"])